In [1]:
import numpy as np # Linear Algebra
import matplotlib.pyplot as plt # Plotting

In [2]:
def RotX(theta):
    ct = np.cos(theta)
    st = np.sin(theta)
    R = np.eye(3, 3)
    R[1, 1] = ct
    R[1, 2] = -st
    R[2, 1] = st
    R[2, 2] = ct
    return R

def RotY(theta):
    ct = np.cos(theta)
    st = np.sin(theta)
    R = np.eye(3, 3)
    R[0, 0] = ct
    R[0, 2] = st
    R[2, 0] = -st
    R[2, 2] = ct
    return R

def RotZ(theta):
    ct = np.cos(theta)
    st = np.sin(theta)
    R = np.eye(3, 3)
    R[0, 0] = ct
    R[0, 1] = -st
    R[1, 0] = st
    R[1, 1] = ct
    return R

In [3]:
def hat(vec):
    v = vec.reshape((3,))
    return np.array([
        [0., -v[2], v[1]],
        [v[2], 0., -v[0]],
        [-v[1], v[0], 0.]
    ])

def angular_velocity_to_deriv_rotation(omega_w, R):
    return hat(omega_w) @ R

def angular_velocity_local_to_deriv_rotation(omega_b, R):
    return R @ hat(omega_b)

In [4]:
# Some testing
R = RotX(np.pi / 2.)

omega_b = np.array([[1., 0., 0.]]).T

# Integrate naively for a few steps
dt = 0.1
R1 = np.copy(R)
R2 = np.copy(R)

for i in range(5):
    omega_w = R1 @ omega_b # rotate local omega to world frame
    R1d = angular_velocity_to_deriv_rotation(omega_w, R1)
    R2d = angular_velocity_local_to_deriv_rotation(omega_b, R2)

    R1 = R1 + R1d * dt # Euler integration!
    R2 = R2 + R2d * dt # Euler integration!

# These two should be very similar
print(R1)
print("========================")
print(R2)
print("========================")
print(np.linalg.norm(R1-R2))

# Let's check if they are good rotation matrices
print("========================")
print(R1@R1.T) # this should be the identity
print(np.linalg.det(R1)) # this should be 1
print("========================")
print(R2@R2.T) # this should be the identity
print(np.linalg.det(R2)) # this should be 1

[[ 1.       0.       0.     ]
 [ 0.      -0.49001 -0.9005 ]
 [ 0.       0.9005  -0.49001]]
[[ 1.       0.       0.     ]
 [ 0.      -0.49001 -0.9005 ]
 [ 0.       0.9005  -0.49001]]
0.0
[[1.         0.         0.        ]
 [0.         1.05101005 0.        ]
 [0.         0.         1.05101005]]
1.0510100500999997
[[1.         0.         0.        ]
 [0.         1.05101005 0.        ]
 [0.         0.         1.05101005]]
1.0510100500999997


In [5]:
# Adjoint Representation
def Adj(R, p):
    Tadj = np.zeros((6, 6))
    Tadj[:3, :3] = R
    Tadj[3:, 3:] = R
    Tadj[3:, :3] = hat(p) @ R

    return Tadj

In [6]:
# Let's do a few tests
Vb = np.array([[1., 2., 0.1, 2., -1., 0.]]).T

R = RotY(np.pi / 2.)
t = np.array([[0., 1., 0.]]).T

Vw = Adj(R, t) @ Vb

Vbn = Adj(R.T, -R.T@t) @ Vw

print(Vb.T)
print(Vw.T)
print(Vbn.T)

[[ 1.   2.   0.1  2.  -1.   0. ]]
[[ 0.1  2.  -1.  -1.  -1.  -2.1]]
[[ 1.00000000e+00  2.00000000e+00  1.00000000e-01  2.00000000e+00
  -1.00000000e+00 -1.75656114e-17]]


In [7]:
def homogeneous(R, p = np.zeros((3, 1))):
    T = np.eye(4)
    T[:3, :3] = R
    T[:3, 3:] = p

    return T

def skew_velocity(V):
    omega = V[:3, :]
    v  = V[3:, :]

    T = np.zeros(shape=(4, 4))
    T[:3, :3] = hat(omega)
    T[:3, 3:] = v

    return T

def spatial_velocity_to_deriv_homogeneous(Vw, T):
    return skew_velocity(Vw) @ T

def spatial_velocity_local_to_deriv_homogeneous(Vb, T):
    return T @ skew_velocity(Vb)

In [8]:
# Let's do full pose integrations
R = RotX(np.pi / 2.)
N_steps = 5

omega_b = np.array([[1., 0., 0.]]).T
v_b = np.array([[0., 1., 0.]]).T

T = homogeneous(R, np.zeros((3, 1)))

Vb = np.vstack([omega_b, v_b])

# Integrate naively for a few steps
dt = 0.1
T1 = np.copy(T)
T2 = np.copy(T)

for i in range(N_steps):
    Vw = Adj(T1[:3, :3], T1[:3, 3:]) @ Vb # rotate local v to world frame
    T1d = spatial_velocity_to_deriv_homogeneous(Vw, T1)
    T2d = spatial_velocity_local_to_deriv_homogeneous(Vb, T2)

    T1 = T1 + T1d * dt # Euler integration!
    T2 = T2 + T2d * dt # Euler integration!

# These two should be very similar
print(T1)
print("========================")
print(T2)
print("========================")
print(np.linalg.norm(T1-T2))
# Let's check if they are good transformation matrices
R1 = T1[:3, :3]
R2 = T2[:3, :3]
print("========================")
print(R1@R1.T) # this should be the identity
print(np.linalg.det(R1)) # this should be 1
print("========================")
print(R2@R2.T) # this should be the identity
print(np.linalg.det(R2)) # this should be 1

[[ 1.       0.       0.       0.     ]
 [ 0.      -0.49001 -0.9005  -0.0995 ]
 [ 0.       0.9005  -0.49001  0.49001]
 [ 0.       0.       0.       1.     ]]
[[ 1.       0.       0.       0.     ]
 [ 0.      -0.49001 -0.9005  -0.0995 ]
 [ 0.       0.9005  -0.49001  0.49001]
 [ 0.       0.       0.       1.     ]]
0.0
[[1.         0.         0.        ]
 [0.         1.05101005 0.        ]
 [0.         0.         1.05101005]]
1.0510100500999997
[[1.         0.         0.        ]
 [0.         1.05101005 0.        ]
 [0.         0.         1.05101005]]
1.0510100500999997


In [11]:
def exp_rotation(p):
    phi = p.reshape((3, 1))
    theta = np.linalg.norm(phi)
    
    if theta < 1e-12:
        return np.eye(3, 3)
    
    a = phi / theta
    R = np.eye(3) * np.cos(theta) + (1. - np.cos(theta)) * a @ a.T + np.sin(theta) * hat(a)

    return R

def log_rotation(R):
    theta = np.arccos(max(-1., min(1., (np.trace(R) - 1.) / 2.)))
    
    if theta < 1e-12:
        return np.zeros((3, 1))
    mat = R - R.T
    r = np.array([mat[2, 1], mat[0, 2], mat[1, 0]]).reshape((3, 1))
    return theta / (2. * np.sin(theta)) * r

def exp_pose(tau):
    theta = np.linalg.norm(tau[:3, :])
    
    R = np.eye(3)
    p = np.zeros((3, 1))

    if not np.isclose(theta, 0.):
        r = tau[:3, :] / theta
        rho = tau[3:, :] / theta
        rh = hat(r)
        R = exp_rotation(tau[:3, :])
        p = (np.eye(3) * theta + (1. - np.cos(theta)) * rh + (theta - np.sin(theta)) * (rh @ rh)) @ rho
    else:
        p = tau[3:, :]
    
    T = np.block([[R, p], [np.zeros((1, 3)), 1.]])

    return T

def log_pose(T):
    R = T[:3, :3]
    p = T[:3, 3:]
    
    rt = log_rotation(R)    
    theta = np.linalg.norm(rt)
    
    if np.allclose(theta, 0.):
        return np.block([[np.zeros((3, 1))], [p]])
    
    rh = hat(rt / theta)

    return np.block([[rt], [(1./theta * np.eye(3) - 0.5 * rh + (1./theta - 0.5 / np.tan(theta/2.)) * (rh @ rh)) @ p * theta]])

Ts = np.array([[10, 13.45, 0, 8.8], [0, -1.23, 0.57, -0.12], [-20.678, 0, -1.2, 92.6], [0, 0, 0, 1]])
print("\nτs =\n", log_pose(Ts))


τs =
 [[ 0.  ]
 [ 0.  ]
 [ 0.  ]
 [ 8.8 ]
 [-0.12]
 [92.6 ]]


In [12]:
# Let's do the integration properly
R = RotX(np.pi / 2.)
N_steps = 5

omega_b = np.array([[1., 0., 0.]]).T
v_b = np.array([[0., 1., 0.]]).T

T = homogeneous(R, np.zeros((3, 1)))

Vb = np.vstack([omega_b, v_b])

# Integrate for a few steps
dt = 0.1
T1 = np.copy(T)
T2 = np.copy(T)

for i in range(N_steps):
    Vw = Adj(T1[:3, :3], T1[:3, 3:]) @ Vb # rotate local v to world frame

    T1 = exp_pose(Vw*dt) @ T1 # exact integration!
    T2 = T2 @ exp_pose(Vb*dt) # exact integration!

# These two should be very similar
print(T1)
print("========================")
print(T2)
print("========================")
print(np.linalg.norm(T1-T2))
# Let's check if they are good transformation matrices
R1 = T1[:3, :3]
R2 = T2[:3, :3]
print("========================")
print(R1@R1.T) # this should be the identity
print(np.linalg.det(R1)) # this should be 1
print("========================")
print(R2@R2.T) # this should be the identity
print(np.linalg.det(R2)) # this should be 1

# Verify that log/exp works as expected!
print("========================")
tau = log_pose(T1)
print("T1 =\n", T1)
print("tau = \n", tau.T)
print("========================")
print(exp_pose(tau))

[[ 1.          0.          0.          0.        ]
 [ 0.         -0.47942554 -0.87758256 -0.12241744]
 [ 0.          0.87758256 -0.47942554  0.47942554]
 [ 0.          0.          0.          1.        ]]
[[ 1.          0.          0.          0.        ]
 [ 0.         -0.47942554 -0.87758256 -0.12241744]
 [ 0.          0.87758256 -0.47942554  0.47942554]
 [ 0.          0.          0.          1.        ]]
5.551115123125783e-17
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
1.0000000000000007
[[1. 0. 0.]
 [0. 1. 0.]
 [0. 0. 1.]]
1.0000000000000007
T1 =
 [[ 1.          0.          0.          0.        ]
 [ 0.         -0.47942554 -0.87758256 -0.12241744]
 [ 0.          0.87758256 -0.47942554  0.47942554]
 [ 0.          0.          0.          1.        ]]
tau = 
 [[2.07079633 0.         0.         0.         0.42120884 0.42120884]]
[[ 1.          0.          0.          0.        ]
 [ 0.         -0.47942554 -0.87758256 -0.12241744]
 [ 0.          0.87758256 -0.47942554  0.47942554]
 [ 0.         